In [1]:
from matplotlib.colors import LinearSegmentedColormap
from datetime import datetime, timedelta
from notebook_utils import calculate_metrics, eval_metrics, timeseries_rel, trim_extremes, catplot_geo
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import geopandas as gpd
import json
import pandas as pd
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Styling Cell
sns.set_theme(context="notebook", style="darkgrid")

SMALL_SIZE = 18
MEDIUM_SIZE = 24
BIGGER_SIZE = 28

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

# To help with visualization, map proper names to the stats
stat_propers = {
    'mae': 'Mean Absolute Error',
    'rmse': 'Root Mean Absolute Error',
    'bias': 'Mean Forecast Bias',
    'corr': 'Correlation Coefficient',
    'skill_score': 'Skill Score'
}

## Import Historical Data

In [13]:
historical = pd.read_csv("../data/monterey_polygon_historical.csv")[["field_id", "crop", "time", "actual_et"]]
historical["time"] = pd.to_datetime(historical["time"])
historical

,field_id,crop,time,actual_et
0,CA_244000,47,2016-01-01,0.554
1,CA_244000,47,2016-01-02,0.713
2,CA_244000,47,2016-01-03,0.810
3,CA_244000,47,2016-01-04,0.579
4,CA_244000,47,2016-01-05,0.631
...,...,...,...,...
2492151,CA_258026,61,2025-05-16,6.513
2492152,CA_258026,61,2025-05-17,6.597
2492153,CA_258026,61,2025-05-18,6.513
2492154,CA_258026,61,2025-05-19,6.931


In [14]:
avgs = pd.read_csv("../data/monterey_polygon_historical_2024_avgs.csv")[["field_id", "crop", "actual_et"]]
avgs

,field_id,crop,actual_et
0,CA_244000,47,2.381216
1,CA_244018,47,1.707648
2,CA_244025,47,2.001825
3,CA_244035,69,2.175964
4,CA_244053,47,1.658634
...,...,...,...
722,CA_257950,47,2.801478
723,CA_257978,47,1.309126
724,CA_257983,47,3.635492
725,CA_258017,47,3.226749


In [15]:
climatology = pd.read_csv("../data/monterey_polygon_historical_climatology.csv")[["field_id", "crop", "doy", "actual_et"]]
climatology

,field_id,crop,doy,actual_et
0,CA_244000,47,1,0.602800
1,CA_244000,47,2,0.646500
2,CA_244000,47,3,0.847200
3,CA_244000,47,4,0.824500
4,CA_244000,47,5,0.720600
...,...,...,...,...
266077,CA_258026,61,362,0.653667
266078,CA_258026,61,363,0.597222
266079,CA_258026,61,364,0.553111
266080,CA_258026,61,365,0.440444


## Import Forecast Data

In [37]:
# Gather current forecast data for the county
forecasts = pd.DataFrame()
files = Path(f"../data/forecasts/fae/monterey/").glob("*.csv")

for file in files:
    parts = str(file.name).split("_")
    data = pd.read_csv(file, low_memory=False)
    data["forecasting_date"] = parts[1].split('.')[0]
    forecasts = pd.concat([data, forecasts], ignore_index=True)

forecasts['forecasting_date'] = pd.to_datetime(forecasts['forecasting_date'])
forecasts['time'] = pd.to_datetime(forecasts['time'])
forecasts

ff = forecasts.loc[:, ['forecasting_date', 'field_id', 'crop', 'time', 'ff_et']]
ff.rename(columns={"ff_et": "et"}, inplace=True)
ff["method"] = "Forward Fill"

wa = forecasts.loc[:, ['forecasting_date', 'field_id', 'crop', 'time', 'avg_et']]
wa.rename(columns={"avg_et": "et"}, inplace=True)
wa["method"] = "Weighted Average Climatology"

med = forecasts.loc[:, ['forecasting_date', 'field_id', 'crop', 'time', 'med_et']]
med.rename(columns={"med_et": "et"}, inplace=True)
med["method"] = "Median Climatology"

forecasts = pd.concat([ff, wa, med], ignore_index=True)
forecasts

,forecasting_date,field_id,crop,time,et,method
0,2025-05-31,CA_244000,47,2025-05-31,3.612,Forward Fill
1,2025-05-31,CA_244000,47,2025-06-01,3.956,Forward Fill
2,2025-05-31,CA_244000,47,2025-06-02,3.642,Forward Fill
3,2025-05-31,CA_244000,47,2025-06-03,3.393,Forward Fill
4,2025-05-31,CA_244000,47,2025-06-04,3.362,Forward Fill
...,...,...,...,...,...,...
15262,2025-05-31,CA_258026,61,2025-06-02,3.757,Median Climatology
15263,2025-05-31,CA_258026,61,2025-06-03,3.723,Median Climatology
15264,2025-05-31,CA_258026,61,2025-06-04,4.345,Median Climatology
15265,2025-05-31,CA_258026,61,2025-06-05,5.567,Median Climatology


## Data Table Creation

In [38]:
data = forecasts.merge(historical, on=["field_id", "crop", "time"], how="left")
data = data.dropna(subset="actual_et")

start_date = data["time"].min()
end_date = data["time"].max()

data

,forecasting_date,field_id,crop,time,et,method,actual_et


## Spatial Plotting

In [39]:
# Import table using only field and hectare columns
field_metadata = pd.read_csv('../data/geo/Monterey_properties.csv')[['field_id', 'hectares']]
# Reformat field IDs to be same convention as the other tables.
field_metadata = field_metadata.set_index('field_id')

# Add additional data to the data table
polygons = pd.read_csv("../data/monterey_polygons.csv", low_memory=False).set_index("OPENET_ID").rename_axis("field_id")

county_line = gpd.read_file('../data/geo/MoCo_Boundary.geojson')
county_line

# Expand .geo column into lon, lat columns
geo = (points[".geo"]
                .apply(lambda x: pd.Series(dict(json.loads(x))))['coordinates']
                .apply(lambda x: pd.Series(list(x), index=['longitude', 'latitude'])))
geo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 739 entries, CA_253578 to CA_251078
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   longitude  739 non-null    float64
 1   latitude   739 non-null    float64
dtypes: float64(2)
memory usage: 17.3+ KB
